# Train New Networks

In our case, this means that everything is converted into `float32` data types, which is necessary for the networks to handle the data efficiently. Neural Network have generally problems processing large numbers, so we take the square root of our trial numbers `.sqrt('num_obs')`. We specify the 'inference_variables' which means that these variables indicate the parameters $\theta$ we want to approximate posteriors $p(\theta | y)$ for. The 'summary_variables' indicate the data that should be summarized by the summary network, so the inference network does not see the raw data but a summarized representation. 'inference_conditions' are additional information that the training is conditioned on, in our case the square root of the trial numbers. As mentioned before, large numbers can cause problems, so we z-standardize our inference variables. We can apply the adapter to our sampled data to check if everything is labelled and transformed correctly:

In [ ]:
adapted_data = adapter(test_data)

for key, i in adapted_data.items():
    print(f'{key}: {i.shape}')

We see that inference_variables include the 1000 parameter draws for each of the six parameters. The summary_variables include 1000 data sets with rt, accuracy and conditions for each of the 200 trials. In the inference_conditions, only 1 value - the number of trials - is stored per data set. 

We define the *summary network* as a set transformer and the *inference network* as flow matching:

In [ ]:

inference_net = bf.networks.FlowMatching(coupling_kwargs=dict(subnet_kwargs=dict(dropout=0.011)))

summary_net = bf.networks.SetTransformer(dropout=0.011, num_seeds=7, summary_dim=22, embed_dim=(128, 128))


In the `workflow` object, we wrap everything up and define some settings for the training phase:

In [ ]:
network_name = 'amortized_dmc'

workflow = bf.BasicWorkflow(
    simulator=simulator,
    adapter=adapter,
    initial_learning_rate=0.00057,
    inference_network=inference_net,
    summary_network=summary_net,
    checkpoint_filepath='../data/training_checkpoints',
    checkpoint_name=network_name,
    inference_variables=param_names,
    save_best_only=True
)


By setting `save_best_only` to `True`, we ensure that the training epoch is only saved if the loss is lower then in the previous epoch. Make sure you specify a valid `checkpoint_filepath`. During the trianing phase, the training checkpoints are going to be stored there and can easily be loaded afterwards.

Before training the network, we simulate an independent sample of data that will later be used to assess potential overfit during the training phase. Since it is not included in the training routine, a larger loss with regards to this data set compared to the training data would indicate an overfit during training. However, since we are following a online training routine and apply multiple methods of regularization (weight decay, dropout and mini-batches), overfitting during the training routine is impossible.

In [ ]:
val_data = simulator.sample(200)

Before starting the training, we have to specify, how long the training should last. The number of epochs indicate how many iterations are included in the training cycle. In each epoch, a specified number of batches will be simulated (`num_batches_per_epoch`) with each batch containing a specified numbers of data sets (`batch_size`). In our case, training will last 200 epochs, simulating 250 batches with 64 data sets in each epochs. All 64 data sets in one batch have the same randomly drawn number of trials.

In [ ]:

history = workflow.fit_online(epochs=200, num_batches_per_epoch=250, batch_size=64, validation_data=val_data)


In [ ]:
figs = workflow.plot_default_diagnostics(test_data=val_data, variable_names=param_labels(param_names), calibration_ecdf_kwargs={'difference': True})
